# 🧪 W12-D4 概念实验：从截图到视频流，差的不只是帧率

配套阅读：`第12周-Day4-VideoAnalytics基线-截图到视频流.md`

md 回答"架构上改什么"，本 notebook 用**可执行实验**回答四个可量化的问题：

1. **采样捕获概率**——事件持续多久、采样间隔多密，才抓得到？（截图模式的数学边界）
2. **算力换算**——A 截图 / B 抽帧 / C 全流 三种方案各需多少推理算力？（1500 倍从哪来）
3. **IoU 跨帧关联**——为什么 Tracking 需要连续帧、截图永远做不了 Tracking？
4. **证据内存预算**——从单张 JPEG 到事件剪辑片段，内存/磁盘预算怎么变？

全部使用 numpy / matplotlib / 标准库，不联网。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

rng = np.random.default_rng(42)
print("字体就绪:", font_name)

## 实验 1：采样捕获概率模型 —— 截图模式的数学边界

**模型**：事件持续时长 D（秒），系统每 T 秒拍一张。事件起始时刻在采样网格上均匀随机。
捕获条件：至少一个采样点落进事件窗口 `[start, start+D)`。

解析解：P(捕获) = min(1, D/T)。**捕获概率只取决于比值 D/T** —— 这就是"业务时间尺度 vs 采样时间尺度"的数学形式。

用 Monte Carlo（20 万次试验）验证解析解，并测出"捕获时的告警时延"。

In [ ]:
N = 200_000

def catch_prob_mc(duration_s, interval_s, n=N):
    """事件时长 duration_s，采样间隔 interval_s，返回(捕获率, 平均告警时延秒)"""
    start = rng.uniform(0.0, interval_s, size=n)          # 事件起点在采样网格上均匀分布
    # 网格采样时刻 k*T；捕获 ⟺ 存在 k 使 start <= kT < start+D
    # 对 0 <= start < T：第一个可能命中的采样点在 [start, start+D) 内 ⟺ ceil(start/T)*T < start+D（k=1 起算即可，因为 start<T）
    next_sample = np.ceil(start / interval_s) * interval_s
    caught = next_sample < start + duration_s
    latency = np.where(caught, next_sample - start, np.nan)
    return caught.mean(), np.nanmean(latency)

scenarios = [
    ("消防通道堆物 30min / 采样 60s",  1800, 60),
    ("火灾早期 90s / 采样 60s(默认)",    90, 60),
    ("火灾早期 90s / 采样 15s(预设)",    90, 15),
    ("老人跌倒 5s / 采样 60s",            5, 60),
    ("老人跌倒 5s / 抽帧 1fps",           5,  1),
    ("老人跌倒 5s / 全流 25fps",          5, 0.04),
]

print(f"{'场景':<28}{'D/T':>8}{'解析解':>10}{'MC仿真':>10}{'平均告警时延':>14}")
rows = []
for name, D, T in scenarios:
    mc, lat = catch_prob_mc(D, T)
    analytic = min(1.0, D / T)
    rows.append((name, analytic, mc, lat))
    print(f"{name:<28}{D/T:>8.2f}{analytic:>9.1%}{mc:>9.1%}{lat:>13.2f}s")

print()
print("结论：D/T>=1 的状态型场景（堆物 30 倍冗余）截图是终局；")
print("      D/T<<1 的事件型场景（跌倒 5s/60s≈8%）截图在数学上就不可行，与模型精度无关。")

In [ ]:
# 可视化：捕获概率曲线 —— 截图(60s)/抽帧(1s)/全流(0.04s) 三种采样密度下的覆盖范围
fig, ax = plt.subplots(figsize=(9, 4.5))
ratio = np.logspace(-2, 1.2, 300)
p = np.minimum(1.0, ratio)
ax.plot(ratio, p, lw=2.5, color="#2563eb", label="P(捕获) = min(1, D/T)")
for D, T, c, lbl in [(5, 60, "#dc2626", "跌倒5s·截图60s"), (5, 1, "#f59e0b", "跌倒5s·抽帧1s"), (5, 0.04, "#16a34a", "跌倒5s·全流25fps")]:
    r, pr = D / T, min(1.0, D / T)
    ax.scatter([r], [pr], color=c, zorder=5, s=70)
    ax.annotate(f"{lbl}\n({pr:.0%})", (r, pr), textcoords="offset points",
                xytext=(12, -12 if pr < 0.5 else -30), fontsize=9, color=c)
ax.set_xscale("log")
ax.set_xlabel("事件时长 / 采样间隔  (D/T，对数轴)")
ax.set_ylabel("捕获概率")
ax.set_title("采样捕获概率：事件越短、采样越稀，截图模式在数学上失效", fontsize=12)
ax.axhline(0.9, ls="--", lw=1, color="gray"); ax.text(0.011, 0.92, "90% 可用线", fontsize=8, color="gray")
ax.grid(alpha=0.3); ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig("/tmp/w12d4_exp1.png", dpi=90); plt.show(); plt.close()

## 实验 2：算力换算 —— "1500 倍"从哪来，三个方案各要什么硬件

MallSenseAI 生产现状：21 路摄像头、CPU 推理（`CUDA_VISIBLE_DEVICES` 留空）。
假设（数量级估算用）：yolo11n CPU 单帧 ≈ 200ms，GPU(RTX 4060) 单帧 ≈ 8ms，服务器 16 核。

In [ ]:
CAMS = 21
CPU_MS, GPU_MS, CORES = 200, 8, 16
plans = [
    ("A 截图(现状) 1张/分钟/路",  1/60),
    ("B RTSP抽帧   1fps/路",      1.0),
    ("C 全流      25fps/路",     25.0),
]

print(f"{'方案':<24}{'总推理/秒':>10}{'需CPU核(200ms/帧)':>18}{'GPU利用率(8ms/帧)':>18}")
for name, fps in plans:
    rate = CAMS * fps
    cpu_cores = rate * CPU_MS / 1000
    gpu_util = rate * GPU_MS / 1000
    print(f"{name:<24}{rate:>10.2f}{cpu_cores:>16.1f}核{gpu_util:>16.1%}")

print()
r_a, r_c = CAMS * (1/60), CAMS * 25
print(f"帧量倍数：全流/截图 = {r_c/r_a:.0f} 倍  (25fps × 60s = 1500，21 路同比例放大)")
print(f"16 核 CPU 推理容量 ≈ {CORES*1000/CPU_MS:.0f} 帧/秒 → 方案A({r_a:.2f}) 富余，方案B({CAMS}) 贴线，方案C({r_c:.0f}) 需 ~{r_c*CPU_MS/1000/CORES:.0f} 台")
print(f"单张 4060 容量 ≈ {1000/GPU_MS:.0f} 帧/秒 → 方案C 裸推理占 1 张卡 {r_c*GPU_MS/1000:.0%}，再叠加 21×3Mbps≈63Mbps 解码与带宽")

fig, ax = plt.subplots(figsize=(9, 4))
names = [p[0] for p in plans]; rates = [CAMS * p[1] for p in plans]
colors = ["#16a34a", "#f59e0b", "#dc2626"]
bars = ax.bar(names, rates, color=colors, width=0.55)
ax.set_yscale("log")
for b, r in zip(bars, rates):
    ax.annotate(f"{r:.2f} 帧/秒", (b.get_x()+b.get_width()/2, r), ha="center", va="bottom", fontsize=10)
ax.axhline(80, ls="--", color="#7c3aed", lw=1.5); ax.text(2.35, 92, "16核CPU容量≈80帧/秒", color="#7c3aed", fontsize=9, ha="right")
ax.axhline(125, ls="--", color="#0891b2", lw=1.5); ax.text(2.35, 142, "单卡4060容量≈125帧/秒", color="#0891b2", fontsize=9, ha="right")
ax.set_ylabel("总推理需求（帧/秒，对数轴）")
ax.set_title("A→B→C 演进谱系的算力账：截图富余、抽帧贴线、全流必上 GPU", fontsize=12)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("/tmp/w12d4_exp2.png", dpi=90); plt.show(); plt.close()

## 实验 3：IoU 跨帧关联 —— 为什么 Tracking 需要连续帧

Tracking（ByteTrack/BoT-SORT 类）的第一步是**帧间关联**：上一帧的检测框和这一帧的检测框算 IoU（交并比），IoU 超过阈值（典型 0.3）才认作同一目标。

构造一个行人穿越画面的合成场景（框 40px、速度 120px/s），观察**采样间隔 Δt 如何摧毁 IoU**：
两帧间位移 d = v·Δt，框边长 s，则 IoU(d) = (s−d)² / (2s² − (s−d)²)（d<s 时，轴对齐同速运动）。

In [ ]:
def iou_consecutive(box_size, speed, dt):
    """同速直线运动下相邻两次观测的 IoU"""
    d = speed * dt
    if d >= box_size:
        return 0.0
    inter = box_size - d
    union = 2 * box_size**2 - inter**2
    return inter**2 / union

s, v = 40.0, 120.0   # 框 40px，行人 120px/s（1080p 画面约 3 秒穿过）
dts = np.logspace(-2, 2.3, 400)          # 0.01s ~ 200s
ious = np.array([iou_consecutive(s, v, t) for t in dts])

# IoU=0.3 的临界采样间隔（数值求根）
from bisect import bisect_left
idx = np.argmin(np.abs(ious[: np.argmax(ious < 0.3)] - 0.3)) if (ious > 0.3).any() else 0
lo, hi = 0.01, 1.0
for _ in range(60):
    mid = (lo + hi) / 2
    if iou_consecutive(s, v, mid) > 0.3: lo = mid
    else: hi = mid
dt_crit = (lo + hi) / 2
print(f"框={s:.0f}px 速度={v:.0f}px/s → IoU≥0.3 的最大采样间隔 ≈ {dt_crit:.2f}s")
print("即：要保住帧间关联，采样必须比 ~{:.1f}s 密。".format(dt_crit))
for label, dt in [("全流 25fps", 0.04), ("抽帧 1fps", 1.0), ("截图 60s", 60.0)]:
    print(f"  {label:<10} Δt={dt:>6.2f}s → IoU={iou_consecutive(s, v, dt):.3f}  {'✅ 可关联' if iou_consecutive(s,v,dt)>=0.3 else '❌ 关联断裂(被视为新目标)'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

# 左图：IoU 随采样间隔衰减
ax = axes[0]
ax.plot(dts, ious, lw=2.5, color="#2563eb")
ax.axhline(0.3, ls="--", color="#dc2626", lw=1.5); ax.text(0.012, 0.34, "关联阈值 0.3", color="#dc2626", fontsize=9)
for label, dt, c in [("25fps", 0.04, "#16a34a"), ("1fps", 1.0, "#f59e0b"), ("截图60s", 60.0, "#dc2626")]:
    y = iou_consecutive(s, v, dt)
    ax.scatter([dt], [y], color=c, s=70, zorder=5)
    ax.annotate(f"{label}\nIoU={y:.2f}", (dt, y), textcoords="offset points", xytext=(10, 8), fontsize=9, color=c)
ax.axvline(dt_crit, ls=":", color="gray")
ax.text(dt_crit*1.15, 0.75, f"临界≈{dt_crit:.1f}s", fontsize=9, color="gray", rotation=90)
ax.set_xscale("log")
ax.set_xlabel("采样间隔 Δt（秒，对数轴）"); ax.set_ylabel("相邻两次观测的 IoU")
ax.set_title("采样越稀，同一行人越像『新目标』", fontsize=11)
ax.grid(alpha=0.3)

# 右图：同一运动，三种采样密度下的"轨迹"
ax = axes[1]
t_end = 3.2
for i, (label, dt, c) in enumerate([("全流25fps", 0.04, "#16a34a"), ("抽帧1fps", 1.0, "#f59e0b"), ("截图60s", 60.0, "#dc2626")]):
    ts = np.arange(0, t_end, dt)
    xs = v * ts
    ax.scatter(ts, np.full_like(ts, 2 - i*0.5), s=14, color=c)
    ax.text(t_end*1.01, 2 - i*0.5, f"{label}：{len(ts)} 个观测点", fontsize=9, va="center", color=c)
ax.set_xlim(-0.1, t_end*1.55)
ax.set_yticks([2, 1.5, 1]); ax.set_yticklabels(["全流", "抽帧", "截图"])
ax.set_xlabel("时间（秒）—— 行人以恒定速度穿越画面 3.2 秒")
ax.set_title("同一目标，三种采样密度看到的『轨迹』", fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout(); plt.savefig("/tmp/w12d4_exp3.png", dpi=90); plt.show(); plt.close()
print("结论：Tracking 的前提是帧间重叠。截图模式(Δt=60s)下每个目标每张图都是『新目标』，")
print("      这不是算法调参能解决的——L2(Video Understanding) 必须以连续帧供给为前提。")

## 实验 4：证据预算 —— 从单张 JPEG 到事件剪辑

截图模式：仅在有检出时落盘一张证据 JPEG（~200KB）。
视频流模式：要留住"事件前 10 秒"，必须常驻**环形缓冲区**（ring buffer），每路持续覆写。

对比三种帧存储形态的预算（21 路）：

In [ ]:
CAMS = 21
pre_s, fps = 10, 25                      # 事件前缓冲 10 秒 @25fps
jpg_frame = 40_000                       # 缓冲帧按 JPEG 压缩 ~40KB
raw_1080p = 1920*1080*3                  # 原始 RGB 一帧 ≈ 5.9MB

rows = [
    ("截图证据（现状）：有检出才存 1 张 JPEG", 200_000 / 1e6, "MB/次", "按事件"),
    (f"环形缓冲@JPEG压缩：{pre_s}s×{fps}fps×40KB×{CAMS}路", pre_s*fps*jpg_frame*CAMS/1e6, "MB 常驻内存", "全时段"),
    (f"环形缓冲@原始帧：{pre_s}s×{fps}fps×5.9MB×{CAMS}路", pre_s*fps*raw_1080p*CAMS/1e9, "GB 常驻内存", "全时段(不可行)"),
]
print(f"{'形态':<44}{'预算':>12}{'单位':<10}")
for name, val, unit, _ in rows:
    print(f"{name:<44}{val:>12.1f}{unit:<10}")

clip_mb = pre_s*3/8                       # 10s×3Mbps H.264 ≈ 3.75MB/事件
alerts_day = 50
print(f"\n事件剪辑落盘：10s×3Mbps≈{clip_mb:.1f}MB/事件，50 事件/天×30 天 ≈ {clip_mb*alerts_day*30/1e3:.1f} GB/月")
print(f"对比截图模式：200KB×50/天×30 天 ≈ {200_000*alerts_day*30/1e9:.2f} GB/月（≈{clip_mb*alerts_day*30/1e3/(200_000*alerts_day*30/1e9):.0f} 倍差距）")
print()
print("结论：1) 环形缓冲必须按压缩帧算（原始帧差 2 个数量级）；")
print("      2) 剪辑证据月增 ~5.6GB/商场，需引入保留策略（如 90 天滚动删除）——截图模式从不面对的存储治理问题。")

## 总结：四个实验拼出的"截图→视频流"完整账本

| 实验 | 回答 | 关键数字 |
|---|---|---|
| 1 采样捕获 | 哪些场景截图数学上可行 | P=min(1, D/T)；跌倒 5s/截图 60s → 8% |
| 2 算力换算 | 三方案各要什么硬件 | 0.35 → 21 → 525 帧/秒；全流≈1500×，CPU 必死 GPU 必选 |
| 3 IoU 关联 | 为什么 Tracking 必须连续帧 | IoU≥0.3 要求 Δt≲0.11s（本实验参数）；截图 60s 下人人都是"新目标" |
| 4 证据预算 | 存储治理的新问题 | 环形缓冲 ~210MB 常驻（JPEG 形态）；剪辑落盘 ~5.6GB/月/商场 |

**架构启示**（对应 md 第 1 节的五层改造）：
- 实验 1 划定**业务边界**——D/T 比值决定场景归 A（截图）还是 C（全流）
- 实验 2 划定**资源边界**——B（RTSP 抽帧 1fps）是唯一不用上 GPU 也能守住 16 核容量的流式方案，且给未来 C 攒流管理经验
- 实验 3 证明**状态层必须新建**——Track 常驻内存不是优化，是 L2 的定义
- 实验 4 证明**证据层必须新建**——环形缓冲 + 保留策略，截图模式的"有检出才落盘"在流模式下失灵